### Stage 1.1: Preparation

In [ ]:
# Install dbt, duckdb, and apache-airflow
!conda install -c conda-forge dbt-core dbt-duckdb apache-airflow -y

In [25]:
# Check if dbt has been successfully installed
!dbt --version

Core:
  - installed: 1.11.7
  - latest:    1.11.7 - Up to date!

Plugins:
  - duckdb: 1.9.4 - Update available!

  At least one plugin is out of date with dbt-core.
  You can find instructions for upgrading here:
  https://docs.getdbt.com/docs/installation




In [40]:
# Check for the path of dbt
import shutil
print(shutil.which('dbt'))

C:\Users\minht\anaconda3\Scripts\dbt.EXE


In [ ]:
# Initialize dbt project (run this in the terminal)
dbt init dbt_project

### Stage 1.2: Configure the Project

`dbt init` created a generic `dbt_project.yml` file for us. Now, we need to make one small edit to tell dbt *how* to build our models.

1.  Navigate to the running folder, click on `dbt_project`.
2.  Click on `dbt_project.yml` to open it in the editor.
3.  Scroll all the way to the bottom to find the `models:` section. It looks like this:
    ```yaml
    models:
      dbt_project:
        # Config indicated by + and applies to all files under models/example/
        example:
          +materialized: view
    ```
4.  **Replace** that *entire* `models:` block with our new rules for `staging`, `intermediate`, and `marts`:
    ```yaml
    models:
      dbt_project:
        staging:
          materialized: view
        intermediate:
          materialized: table
        marts:
          materialized: view
    ```

### Stage 1.3: Configure the Connection Profile

To tell dbt **how** and **where** to connect to our data (the DuckDB file), we need to create the global `profiles.yml` file. This is the **connection string** that dbt will use for every subsequent command.

Hence, we need to define a profile named `jaffle_shop_duckdb` which points directly to our local DuckDB file path.

In [46]:
import os

# Ensure the .dbt directory exists
os.makedirs(os.path.expanduser("~/.dbt"), exist_ok=True)

# Now write the profiles.yml file
with open(os.path.expanduser("~/.dbt/profiles.yml"), "w") as f:
    f.write("""jaffle_shop_duckdb:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: dev.duckdb
      threads: 1
""")

### Stage 2: Load (Seed) Raw Data (The "L" in ELT)

Great! The project is now fully configured.

Now, we Copy our *real* data (the Jaffle Shop CSVs) into the `seeds` folder and run `dbt seed`.

#### Why We Use `dbt seed` here:

While seeds are typically used in production for **small, static data** that rarely changes, we use them here for **convenience** and **replicability**. This step:
* **1. Creates Base Tables:** It quickly loads the raw CSV data into queryable tables within our local DuckDB database.
* **2. Guarantees Consistency:** It ensures that every participant starts with the **exact same raw data**, making our downstream tests and transformations reliable.

In [47]:
# 1. Copy our real data into the seeds folder. We can do it manually or run the following code
!cp raw_customers.csv dbt_project/seeds/
!cp raw_orders.csv dbt_project/seeds/
!cp raw_payments.csv dbt_project/seeds/

'cp' is not recognized as an internal or external command,
operable program or batch file.
'cp' is not recognized as an internal or external command,
operable program or batch file.
'cp' is not recognized as an internal or external command,
operable program or batch file.


In [48]:
# 2. Run dbt seed!
!dbt seed --project-dir jaffle_shop_duckdb --profile jaffle_shop_duckdb

13:34:46  Running with dbt=1.11.7
13:34:46  Registered adapter: duckdb=1.9.4
13:34:47  Found 7 models, 3 seeds, 4 data tests, 3 sources, 429 macros
13:34:47  
13:34:47  Concurrency: 1 threads (target='dev')
13:34:47  
13:34:47  1 of 3 START seed file main.raw_customers ...................................... [RUN]
13:34:47  1 of 3 OK loaded seed file main.raw_customers .................................. [INSERT 100 in 0.09s]
13:34:47  2 of 3 START seed file main.raw_orders ......................................... [RUN]
13:34:47  2 of 3 OK loaded seed file main.raw_orders ..................................... [INSERT 99 in 0.04s]
13:34:47  3 of 3 START seed file main.raw_payments ....................................... [RUN]
13:34:48  3 of 3 OK loaded seed file main.raw_payments ................................... [INSERT 113 in 0.13s]
13:34:48  
13:34:48  Finished running 3 seeds in 0 hours 0 minutes and 0.70 seconds (0.70s).
13:34:48  
13:34:48  Completed successfully
13:34:48  
13:34

## Stage 3: Build Transformation Layer (The "T" in ELT)

Next, we build the **Transformation Layer** of our pipeline  staging, intermediate, and marts models . This is where we apply business logic to clean, combine, and reshape the raw data into valuable, analytics-ready tables.

This process follows a standard data modeling approach:
* **Sources:** Defines and documents the raw, external data tables (the entry point to the transformation pipeline).
* **Staging:** Cleans and renames raw source data (simple 1:1 views).
* **Intermediate:** Joins and aggregates data (performs complex logic).
* **Marts:** Creates final, clean tables ready for BI tools.

### 3.1 Sources
A source file is created, which tells dbt where to find the raw data tables (the seeds we loaded in Stage 2) in the database.

In [52]:
%%writefile jaffle_shop_duckdb/models/staging/source.yml
version: 2

sources:
  - name: jaffle_shop_duckdb
    schema: main # This is the schema DuckDB created for our seeds
    tables:
      - name: raw_customers
      - name: raw_orders
      - name: raw_payments

Overwriting jaffle_shop_duckdb/models/staging/source.yml


### 3.2 Staging Layer
At this step, we want to create files where we store SQL commands that perform light tasks such as selecting columns and renaming their names.

In [ ]:
# Create the staging folder
!mkdir -p jaffle_shop_duckdb/models/staging

In [53]:
%%writefile jaffle_shop_duckdb/models/staging/stg_customers.sql
select
    id as customer_id,
    first_name,
    last_name
from {{ source('jaffle_shop_duckdb', 'raw_customers') }}

Overwriting jaffle_shop_duckdb/models/staging/stg_customers.sql


In [54]:
%%writefile jaffle_shop_duckdb/models/staging/stg_orders.sql
select
    id as order_id,
    user_id as customer_id,
    order_date,
    status
from {{ source('jaffle_shop_duckdb', 'raw_orders') }}

Overwriting jaffle_shop_duckdb/models/staging/stg_orders.sql


In [55]:
%%writefile jaffle_shop_duckdb/models/staging/stg_payments.sql
select
    id as payment_id,
    order_id,
    amount,
    payment_method
from {{ source('jaffle_shop_duckdb', 'raw_payments') }}

Overwriting jaffle_shop_duckdb/models/staging/stg_payments.sql


In [56]:
# Run dbt run to execute all the models in our staging folder (stg_customers, stg_orders, stg_payments)
!dbt run --select staging --project-dir jaffle_shop_duckdb --profile jaffle_shop_duckdb

13:36:49  Running with dbt=1.11.7
13:36:50  Registered adapter: duckdb=1.9.4
13:36:51  Found 7 models, 3 seeds, 4 data tests, 3 sources, 429 macros
13:36:51  
13:36:51  Concurrency: 1 threads (target='dev')
13:36:51  
13:36:51  1 of 3 START sql view model main.stg_customers ................................. [RUN]
13:36:51  1 of 3 OK created sql view model main.stg_customers ............................ [OK in 0.12s]
13:36:51  2 of 3 START sql view model main.stg_orders .................................... [RUN]
13:36:51  2 of 3 OK created sql view model main.stg_orders ............................... [OK in 0.14s]
13:36:51  3 of 3 START sql view model main.stg_payments .................................. [RUN]
13:36:51  3 of 3 OK created sql view model main.stg_payments ............................. [OK in 0.05s]
13:36:51  
13:36:51  Finished running 3 view models in 0 hours 0 minutes and 0.70 seconds (0.70s).
13:36:52  
13:36:52  Completed successfully
13:36:52  
13:36:52  Done. PASS=3

### 3.3 Intermediate Layer
Now, we move to the intermediate layer where we perform join between the previously built models and some heavier calculations. 

In [ ]:
!mkdir -p jaffle_shop_duckdb/models/intermediate

The syntax of the command is incorrect.


In [58]:
%%writefile jaffle_shop_duckdb/models/intermediate/int_orders_enriched.sql
with orders as (
 select * from {{ ref('stg_orders') }}
),
payments as (
 select * from {{ ref('stg_payments') }}
),
order_payments as (
 select
 orders.order_id,
 orders.customer_id,
 orders.order_date,
 orders.status,
 payments.amount
 from orders
 left join payments 
    on orders.order_id = payments.order_id
),
final_payments as (
 select
 order_id,
 customer_id,
 order_date,
 sum(case when trim(status) = 'completed' then amount else 0 end) as amount
 from order_payments
 group by 1, 2, 3
)
select * from final_payments

Overwriting jaffle_shop_duckdb/models/intermediate/int_orders_enriched.sql


In [59]:
!dbt run --select int_orders_enriched --project-dir jaffle_shop_duckdb --profile jaffle_shop_duckdb

13:37:20  Running with dbt=1.11.7
13:37:21  Registered adapter: duckdb=1.9.4
13:37:21  Found 7 models, 3 seeds, 4 data tests, 3 sources, 429 macros
13:37:21  
13:37:21  Concurrency: 1 threads (target='dev')
13:37:21  
13:37:22  1 of 1 START sql table model main.int_orders_enriched .......................... [RUN]
13:37:22  1 of 1 OK created sql table model main.int_orders_enriched ..................... [OK in 0.20s]
13:37:22  
13:37:22  Finished running 1 table model in 0 hours 0 minutes and 0.59 seconds (0.59s).
13:37:22  
13:37:22  Completed successfully
13:37:22  
13:37:22  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=1


### 3.4 Final Layer (Marts Model)
Moving to the final layer! Marts models are the clean, end-user-facing models that power dashboards and analysis. They are built from the previous staging and intermediate models.

We'll build two:

+ fct_orders.sql: A "fact" table that joins our int_orders model with stg_customers.
+ dim_customers.sql: A "dimension" table that is just a clean copy of stg_customers.

In [ ]:
!mkdir -p jaffle_shop_duckdb/models/mart

In [60]:
%%writefile jaffle_shop_duckdb/models/marts/fct_orders.sql
with orders as (
    select * from {{ ref('int_orders_enriched') }}
),

customers as (
    select * from {{ ref('stg_customers') }}
),

final as (
    select
        orders.order_id,
        orders.customer_id,
        customers.first_name,
        customers.last_name,
        orders.order_date,
        orders.amount
    from orders
    left join customers using (customer_id)
)

select * from final

Overwriting jaffle_shop_duckdb/models/marts/fct_orders.sql


In [61]:
%%writefile jaffle_shop_duckdb/models/marts/dim_customers.sql
select
    customer_id,
    first_name,
    last_name
from {{ ref('stg_customers') }}

Overwriting jaffle_shop_duckdb/models/marts/dim_customers.sql


In [62]:
!dbt run --select marts --project-dir jaffle_shop_duckdb --profile jaffle_shop_duckdb

13:37:52  Running with dbt=1.11.7
13:37:53  Registered adapter: duckdb=1.9.4
13:37:53  Found 7 models, 3 seeds, 4 data tests, 3 sources, 429 macros
13:37:53  
13:37:53  Concurrency: 1 threads (target='dev')
13:37:53  
13:37:54  1 of 2 START sql view model main.dim_customers ................................. [RUN]
13:37:54  1 of 2 OK created sql view model main.dim_customers ............................ [OK in 0.11s]
13:37:54  2 of 2 START sql view model main.fct_orders .................................... [RUN]
13:37:54  2 of 2 OK created sql view model main.fct_orders ............................... [OK in 0.20s]
13:37:54  
13:37:54  Finished running 2 view models in 0 hours 0 minutes and 0.77 seconds (0.77s).
13:37:54  
13:37:54  Completed successfully
13:37:54  
13:37:54  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=2


### Stage 4: Test Our Models
This is a critical step in any dbt project. We need to add data tests to ensure our logic is correct and the data is clean.

We'll add a new schema.yml file in our marts folder to define some tests:

+ dim_customers should have a customer_id that is both unique and never null.
+ fct_orders should have an order_id that is unique and never null.
+ Then, we'll run dbt test.

In [63]:
%%writefile jaffle_shop_duckdb/models/marts/schema.yml
version: 2

models:
  - name: dim_customers
    columns:
      - name: customer_id
        tests:
          - unique
          - not_null

  - name: fct_orders
    columns:
      - name: order_id
        tests:
          - unique
          - not_null

Overwriting jaffle_shop_duckdb/models/marts/schema.yml


In [64]:
!dbt test --project-dir jaffle_shop_duckdb --profile jaffle_shop_duckdb

13:38:09  Running with dbt=1.11.7
13:38:09  Registered adapter: duckdb=1.9.4
13:38:10  Found 7 models, 3 seeds, 4 data tests, 3 sources, 429 macros
13:38:10  
13:38:10  Concurrency: 1 threads (target='dev')
13:38:10  
13:38:11  1 of 4 START test not_null_dim_customers_customer_id ........................... [RUN]
13:38:11  1 of 4 PASS not_null_dim_customers_customer_id ................................. [PASS in 0.06s]
13:38:11  2 of 4 START test not_null_fct_orders_order_id ................................. [RUN]
13:38:11  2 of 4 PASS not_null_fct_orders_order_id ....................................... [PASS in 0.11s]
13:38:11  3 of 4 START test unique_dim_customers_customer_id ............................. [RUN]
13:38:11  3 of 4 PASS unique_dim_customers_customer_id ................................... [PASS in 0.03s]
13:38:11  4 of 4 START test unique_fct_orders_order_id ................................... [RUN]
13:38:11  4 of 4 PASS unique_fct_orders_order_id ........................

### Stage 5: Orchestrate The Pipeline with Airflow

In [ ]:
# This command initializes the Airflow metadata database
!airflow db migrate

In [ ]:
!mkdir dags

A subdirectory or file dags already exists.


In [75]:
%%writefile dags/dbt_pipeline_dag.py
from airflow.decorators import dag
from airflow.providers.standard.operators.bash import BashOperator
from pendulum import datetime
import os

# 1. DEFINITIONS (Absolute WSL paths)
WORKSPACE_ROOT = "/mnt/c/Users/minht/Downloads/Customers_Insight_Pipeline"
DBT_PROJECT_DIR = f"{WORKSPACE_ROOT}/jaffle_shop_duckdb"
# Use the full path to the dbt executable inside your venv
DBT_BIN = f"{WORKSPACE_ROOT}/venv/bin/dbt"

@dag(
    dag_id="dbt_jaffle_shop_pipeline",
    start_date=datetime(2024, 1, 1),
    schedule="@daily",
    catchup=False,
)
def dbt_jaffle_shop_dag():

    dbt_build_task = BashOperator(
        task_id="dbt_build_pipeline",
        bash_command=f"""
        set -e
        
        # Move to the project root so 'cp' works
        cd {WORKSPACE_ROOT}
        
        # Copy seeds using absolute paths
        cp {WORKSPACE_ROOT}/raw_customers.csv {DBT_PROJECT_DIR}/seeds/
        cp {WORKSPACE_ROOT}/raw_orders.csv {DBT_PROJECT_DIR}/seeds/
        cp {WORKSPACE_ROOT}/raw_payments.csv {DBT_PROJECT_DIR}/seeds/
        
        # Run dbt with absolute binary path and project dir
        {DBT_BIN} build --project-dir {DBT_PROJECT_DIR} --profiles-dir /home/triet/.dbt
        """
    )

    dbt_build_task

dbt_jaffle_shop_dag()

Overwriting dags/dbt_pipeline_dag.py


In [ ]:
# Close the previous Python connection to free the database lock ---
try:
    if 'conn' in locals() and conn:
        conn.close()
    print("Database connection closed successfully.")
except NameError:
    pass
    
# Tell Airflow to use *only* our new 'dags' folder
%env AIRFLOW__CORE__DAGS_FOLDER=dags

Database connection closed successfully.
env: AIRFLOW__CORE__DAGS_FOLDER=dags


In [69]:
# Run the single dbt_build_pipeline task
print("--- RUNNING FULL PIPELINE: dbt_build_pipeline ---")
!dbt build --project-dir jaffle_shop_duckdb --profile jaffle_shop_duckdb

--- RUNNING FULL PIPELINE: dbt_build_pipeline ---
13:45:04  Running with dbt=1.11.7
13:45:05  Registered adapter: duckdb=1.9.4
13:45:06  Found 7 models, 3 seeds, 4 data tests, 3 sources, 429 macros
13:45:06  
13:45:06  Concurrency: 1 threads (target='dev')
13:45:06  
13:45:06  1 of 14 START sql view model main.stg_customers ................................ [RUN]
13:45:06  1 of 14 OK created sql view model main.stg_customers ........................... [OK in 0.21s]
13:45:06  2 of 14 START sql view model main.stg_orders ................................... [RUN]
13:45:06  2 of 14 OK created sql view model main.stg_orders .............................. [OK in 0.04s]
13:45:06  3 of 14 START sql view model main.stg_payments ................................. [RUN]
13:45:06  3 of 14 OK created sql view model main.stg_payments ............................ [OK in 0.04s]
13:45:06  4 of 14 START seed file main.raw_customers ..................................... [RUN]
13:45:06  4 of 14 OK loaded s

In [ ]:
# Run this (with airflow)
print("--- RUNNING FULL PIPELINE: dbt_build_pipeline ---")
!airflow tasks test dbt_jaffle_shop_pipeline dbt_build_pipeline 2024-01-01

### Stage 6: The Final Analysis
The entire pipeline—from raw data to tested transformations—was created to answer the question, "Who are my most valuable customers?" The answer is found in the final mart table, dim_customers.

We'll use a Python query to select and display the top 10 most valuable customers (based on order amount).

In [73]:
import duckdb

# 1. Update the path to the absolute WSL location
DB_PATH = r'C:\Users\minht\Downloads\Customers_Insight_Pipeline\dev.duckdb'

# 2. Connect using the correct path
conn = duckdb.connect(database=DB_PATH, read_only=True)

# The rest of your query remains the same
top_customers_query = """
SELECT 
    c.customer_id,
    c.first_name,
    c.last_name,
    SUM(f.amount) AS total_lifetime_value
FROM dim_customers AS c
JOIN fct_orders AS f ON c.customer_id = f.customer_id
GROUP BY 1, 2, 3
ORDER BY total_lifetime_value DESC
LIMIT 10
"""

# Execute and display
df_final = conn.execute(top_customers_query).fetchdf()
print("Top 10 Most Valuable Customers (Final Output):")
display(df_final)

Top 10 Most Valuable Customers (Final Output):


,customer_id,first_name,last_name,total_lifetime_value
0,51,Howard,R.,9900.0
1,3,Kathleen,P.,6500.0
2,50,Billy,L.,4700.0
3,8,Frank,R.,4500.0
4,99,Mary,G.,4400.0
5,71,Gerald,C.,4400.0
6,54,Rose,M.,4100.0
7,53,Anne,B.,3900.0
8,69,Janet,P.,3200.0
9,32,Thomas,O.,3000.0


In [74]:
import duckdb
DB_PATH = r'C:\Users\minht\Downloads\Customers_Insight_Pipeline\dev.duckdb'
conn = duckdb.connect(database=DB_PATH, read_only=True)

# This will list every table and which schema it belongs to
print(conn.execute("SELECT table_schema, table_name FROM information_schema.tables").df())

  table_schema           table_name
0         main           int_orders
1         main  int_orders_enriched
2         main        raw_customers
3         main           raw_orders
4         main         raw_payments
5         main        dim_customers
6         main           fct_orders
7         main        stg_customers
8         main           stg_orders
9         main         stg_payments
